In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
print(sys.executable)

X_df = pd.read_pickle('../data/processed/X_df.pkl')
y_df = pd.read_pickle('../data/processed/y_df.pkl')

/Users/shanewarland/miniforge3/envs/protein-ml/bin/python


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = X_df['WT_name']

splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)


train_idx, test_idx = next(splitter.split(X_df, y_df, groups=groups))

X_train, X_test  = X_df.iloc[train_idx].copy(), X_df.iloc[test_idx].copy()
y_train, y_test = y_df.iloc[train_idx].copy(), y_df.iloc[test_idx].copy()
 
print(X_train.columns)
print(X_train.head(2))

Adding some additional biochemical features like molecular weight, hydrophobicity, amino acid charge, polarity, and if aromatic

In [36]:
from src.features import get_biochemical_features

X_train = get_biochemical_features(X_train)
X_test = get_biochemical_features(X_test)

## Grab numeric columns
feature_cols = ["position", "relative_position", "protein_length",
    "wt_hydrophobicity", "mut_hydrophobicity", "delta_hydrophobicity", 
    "wt_mw", "mut_mw","delta_mw", 
    "wt_charge", "mut_charge", "delta_charge", "wt_aromatic", "mut_aromatic"]


First we will use basic linear models to see how these compare. Because many of these features are related, we use ridge which applies an L2 penalty.

In [23]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baseline = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])

baseline.fit(X_train[feature_cols], y_train)

y_pred = baseline.predict(X_test[feature_cols])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(y_test, y_pred) ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

from scipy.stats import spearmanr

rho, p_value = spearmanr(y_test, y_pred)

print("Spearman rho:", rho)
print("p-value:", p_value)


MAE: 0.7041848078174705
RMSE: 0.9079423791122039
R²: 0.1751367610335689
Spearman rho: 0.39220526831229097
p-value: 0.0


Linear models don't do a great job at predicting protein stability. Only 17% of variance in delta delta R was explained by the model variables. Low correlation between Y_pred and Y_test. Signficant, but there are 76k samples in y_test so not really useful.

Random forest might be able to improve accuracy a bit. String classifing variables like aromatic can be added. 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_model.fit(X_train[feature_cols], y_train)

# 5. Make predictions on the unseen test data
y_pred = rf_model.predict(X_test[feature_cols])


mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(y_test, y_pred) ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

rho, p = spearmanr(y_test, y_pred)

print("Spearman rho:", rho)


MAE: 0.5941913501956142
RMSE: 0.8335679238282023
R²: 0.3047398103824638


Random Forest performed slightly better with lower error and higher variance explained (R² = 0.30). This supports the hypothesis that the relationship between amino acid mutations and protein stability is nonlinear. Spearman rho is 0.59 so weak positive relationshp. 

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df

So relative position of the mutations is most important. This makes sense as a C terminal mutation probaly is less likely to destabilize protein compared to one in the center. 

Can we tune the model and get a bit more out of it?

In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV

rf_model = RandomForestRegressor(random_state=42, n_jobs=1)

## This ensures that 
cv = GroupKFold(n_splits=5)

param_grid = {
    "n_estimators": [100, 200],
    "max_features": ["sqrt", 0.5, 1.0],
    "min_samples_leaf": [1, 10, 50],
}

grid_search = GridSearchCV(estimator=rf_model, 
    param_grid=param_grid, 
    cv=cv, 
    scoring="neg_mean_absolute_error", 
    n_jobs=-1, verbose = 2)


grid_search.fit(X_train[feature_cols], y_train, groups=X_train["WT_name"])


best_rf = grid_search.best_estimator_
print("\nBest Parameters Found:")
print(grid_search.best_params_)
#Best Parameters Found:
#{'max_features': 0.5, 'min_samples_leaf': 10, 'n_estimators': 200}

y_pred = best_rf.predict(X_test[feature_cols])
mae = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
rho, p = spearmanr(y_test, y_pred)
print("Spearman rho:", rho)

#MAE: 0.5941913501956142
#RMSE: 0.7902706549331682
#R²: 0.3750905603618625
#Spearman rho: 0.6031151767936802

Ok so tuning the random forest caused some improvement, (R² = 0.37). Rho changed very little. Let's try XGboost. Need to be careful with tuning due to compute contraints. I'll use a RandomizedSearchCV to help speed things up. 

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_model = XGBRegressor(random_state=42, objective="reg:squarederror", n_jobs=1)

## This ensures that 
cv = GroupKFold(n_splits=5)

param_grid = {
    "n_estimators": [200, 500, 800],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
}


grid_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=25,         
    scoring="neg_mean_absolute_error",
    cv=cv,          
    verbose=2,
    random_state=42,
    n_jobs=-1)

grid_search.fit(X_train[feature_cols], y_train, groups=X_train["WT_name"])

best_xgb = grid_search.best_estimator_
print("\nBest Parameters Found:")
print(grid_search.best_params_)
#Best Parameters Found:
#{'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.9}


y_pred = best_xgb.predict(X_test[feature_cols])
mae = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
rho, p = spearmanr(y_test, y_pred)
print("Spearman rho:", rho)

#MAE: 0.6518394509782319
#RMSE: 0.807365747959518
#R²: 0.347762123609085
#Spearman rho: 0.5843680407647296

Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best Parameters Found:
{'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
MAE: 0.5941913501956142
RMSE: 0.807365747959518
R²: 0.347762123609085
Spearman rho: 0.5843680407647296


Ok still did not improve over RF but had very similar performances. I held back on tuning a bit due to compute contraints. Both tree models seems to plateau around R² = 0.34-0.4. There probably is not enough information encoded within my selected features to improve further. This will be interesting to see how a neural net can improve on this. 

Begin building neural net in notebook 3.